# DQ 检查 — DWS 层（Rule 13）

Rule 13: 聚合守恒（DWS trip_count ≈ fact 行数）

FAIL 将抛出异常，阻断 ADS 下游任务。

## 1. SparkSession

In [1]:
import os
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("TLC-DQ-AfterDWS")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE     = os.path.expanduser("~/tlc-spark-pipeline")
DWD_FACT = f"{BASE}/data/warehouse/dwd/fact_trips"
DWS_OUT  = f"{BASE}/data/warehouse/dws/trips_daily"

TARGET_MONTH = os.environ.get("TARGET_MONTH", "2024-03")
yr, mo = int(TARGET_MONTH.split("-")[0]), int(TARGET_MONTH.split("-")[1])
print(f"TARGET_MONTH={TARGET_MONTH}")

print(f"Spark {spark.version} ready | TARGET_MONTH={TARGET_MONTH}")


## 2. Rule 13：DWS 聚合守恒

In [2]:
fact_count     = spark.read.parquet(DWD_FACT).filter(
    (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo)).count()
dws            = spark.read.parquet(DWS_OUT).filter(
    (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo))
dws_total      = dws.agg(F.sum("trip_count")).collect()[0][0]
diff_rate      = abs(dws_total - fact_count) / fact_count

passed = diff_rate < 0.001
mark   = "✅" if passed else "❌"
print(f"{mark} Rule 13 [DWS] DWS trips ≈ fact rows")
print(f"   fact_count={fact_count:,}  dws_total={dws_total:,}  diff={diff_rate:.4%}")

print(f"\n{'='*50}")
print(f"DQ Report (DWS)  {datetime.now().strftime('%Y-%m-%d %H:%M')}")

spark.stop()

if not passed:
    raise ValueError(
        f"DQ FAILED Rule 13: DWS aggregation mismatch {diff_rate:.4%}. "
        "Pipeline halted — fix DWS before proceeding to ADS."
    )
else:
    print("Rule 13 passed — proceeding to ADS.")

print("DQ after-DWS check complete.")
